# 06 · Reproducción del champion desde el Model Registry

Notebook limpio: no entrena nada. Carga `models:/ecu911-modelo@champion`, reconstruye el hold-out con el
mismo código (`src/datos.py`) y el mismo snapshot (se verifica el hash) y reproduce **exactamente** la
métrica registrada en la corrida final.

In [ ]:
import sys, warnings
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings("ignore")
import src  # noqa: F401  (en Windows carga LightGBM antes que pyarrow)
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from src.datos import *
FIG = RAIZ / "docs" / "figuras"; FIG.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

In [ ]:
import mlflow
from mlflow import MlflowClient
from src.experimento import configurar, MODELO_REGISTRO
from src.pipeline import pr_auc, costo, ReglaEstacional, metricas_negocio
configurar(); client = MlflowClient()

version = client.get_model_version_by_alias(MODELO_REGISTRO, "champion")
run = client.get_run(version.run_id)
print(f"{MODELO_REGISTRO} v{version.version} (alias champion) ← run {version.run_id} [{run.data.tags['modelo']}]")
registrado = run.data.metrics
umbral = float(run.data.params["umbral_decision"])

In [ ]:
obt = cargar_obt()
md_ = meta(obt)
assert md_.hash == run.data.params["datos_hash"], "El snapshot de datos no es el mismo que se usó en el registro"
print("Hash del snapshot coincide:", md_.hash)
_, test = dividir(modelable(construir_dataset(obt)))
assert str(test.fecha.min().date()) == run.data.params["test_desde"] and len(test) == int(run.data.params["n_test"])

modelo = mlflow.sklearn.load_model(f"models:/{MODELO_REGISTRO}@champion")
score = modelo.predict_proba(test[FEATURES])[:, 1]
reproducido = {"test_pr_auc": pr_auc(test.y, score)}
nb_ = metricas_negocio(test.y.values, score, umbral, ReglaEstacional().predict(test[FEATURES]))
reproducido.update({k: nb_[k] for k in ["reduccion_fn_pct", "costo_modelo_por_1000", "recall_umbral", "precision_umbral"]})
tabla = pd.DataFrame({"registrado": {k: registrado[k] for k in reproducido}, "reproducido": reproducido})
tabla["diferencia"] = tabla.reproducido - tabla.registrado
display(tabla)
assert (tabla.diferencia.abs() < 1e-9).all()
print("OK: el modelo champion reproduce exactamente las métricas registradas.")

In [ ]:
# También vía pyfunc (como se serviría en el PSet 4): misma salida que predict_proba[:, 1]
pyf = mlflow.pyfunc.load_model(f"models:/{MODELO_REGISTRO}@champion")
print("Firma de entrada:", pyf.metadata.get_input_schema().input_names()[:6], "...")
salida = np.asarray(pyf.predict(test[FEATURES].head(1000)))[:, 1]
print("max |pyfunc - predict_proba| =", np.abs(salida - score[:1000]).max())
alerta = salida >= umbral
print(f"Alertas en las primeras 1000 filas con el umbral de negocio {umbral:.4f}: {alerta.sum()}")